In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('awards.csv', encoding='latin1')
df

In [ ]:
df[df['Baccalaureate Institution'] == 'University of Delaware']

In [ ]:
chemical_engineering_seniors = df.loc[
    (df['Field of Study'] == 'Engineering - Chemical Engineering')
    & (df['Current Institution'].isna()),
    ['Baccalaureate Institution']
]

baccalaureate_counts = (
    chemical_engineering_seniors
    .value_counts()
    .rename_axis('Baccalaureate Institution')
    .reset_index(name='Number of Awardees')
)
baccalaureate_counts.to_markdown('chemical_engineering_seniors_by_baccalaureate.md', index=False)
baccalaureate_counts


In [ ]:
same_institution = df.loc[
    (df['Field of Study'] == 'Engineering - Chemical Engineering')
    & (df['Current Institution'] == df['Baccalaureate Institution']),
    ['Baccalaureate Institution']
]

same_institution_counts = (
    same_institution
    .value_counts()
    .rename_axis('Baccalaureate Institution')
    .reset_index(name='Number of Awardees')
)
same_institution_counts.to_markdown('chemical_engineering_same_institution_by_baccalaureate.md', index=False)
print(f"total # awardees: {same_institution_counts['Number of Awardees'].sum()}")
same_institution_counts


In [ ]:
field_award_breakdown = (
    df.assign(
        senior_undergrad=df['Current Institution'].eq(df['Baccalaureate Institution']),
        first_year_grad_students_ish=(
            df['Current Institution'].notna()
            & df['Current Institution'].ne(df['Baccalaureate Institution'])
        ),
    )
    .groupby('Field of Study', dropna=False)
    .agg(
        total_awards=('Name', 'size'),
        senior_undergrads=('senior_undergrad', 'sum'),
        first_year_grad_students_ish=('first_year_grad_students_ish', 'sum'),
    )
    .reset_index()
    .sort_values(['total_awards', 'Field of Study'], ascending=[False, True])
)

field_award_breakdown = field_award_breakdown.rename(columns={
    'Field of Study': 'Field of Study',
    'total_awards': 'Total Awards',
    'senior_undergrads': 'Senior Undergrads',
    'first_year_grad_students_ish': 'First-Year Grad Students-ish',
})
field_award_breakdown.to_markdown('field_of_study_award_breakdown.md', index=False)
field_award_breakdown


In [ ]:
chemical_engineering_first_year_phd_winners = df.loc[
    (df['Field of Study'] == 'Engineering - Chemical Engineering')
    & df['Current Institution'].notna()
    & df['Current Institution'].ne(df['Baccalaureate Institution']),
    ['Current Institution']
]

chemical_engineering_first_year_phd_by_school = (
    chemical_engineering_first_year_phd_winners
    .value_counts()
    .rename_axis('Current Institution')
    .reset_index(name='Number of First-Year PhD Student Winners')
    .sort_values(['Number of First-Year PhD Student Winners', 'Current Institution'], ascending=[False, True])
)
chemical_engineering_first_year_phd_by_school.to_markdown('chemical_engineering_first_year_phd_by_current_institution.md', index=False)
chemical_engineering_first_year_phd_by_school


## Publication visualizations

Generate the 600-DPI Okabe–Ito figures used in the Quarto report.

In [ ]:
from pathlib import Path
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


DATA_PATH = Path("awards.csv")
FIGURE_DIR = Path("figures")
FIELD = "Engineering - Chemical Engineering"

COLORS = {
    "orange": "#E69F00",
    "sky_blue": "#56B4E9",
    "bluish_green": "#009E73",
    "yellow": "#F0E442",
    "blue": "#0072B2",
    "vermillion": "#D55E00",
    "reddish_purple": "#CC79A7",
    "black": "#000000",
}


def wrap_labels(values, width=34):
    return [textwrap.fill(str(value), width=width) for value in values]


def save_figure(fig, path):
    fig.savefig(
        path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white",
    )


def style_axis(ax, axis="x"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis=axis, color="#D9E2E8", linewidth=0.7, alpha=0.8)
    ax.set_axisbelow(True)


def field_breakdown(df):
    return (
        df.assign(
            same_institution=df["Current Institution"].eq(
                df["Baccalaureate Institution"]
            ),
            different_institution=(
                df["Current Institution"].notna()
                & df["Current Institution"].ne(df["Baccalaureate Institution"])
            ),
        )
        .groupby("Field of Study", dropna=False)
        .agg(
            total_awards=("Name", "size"),
            same_institution=("same_institution", "sum"),
            different_institution=("different_institution", "sum"),
        )
        .reset_index()
        .sort_values(["total_awards", "Field of Study"], ascending=[False, True])
    )


FIGURE_DIR.mkdir(exist_ok=True)
visualization_field_counts = field_breakdown(df)


### Fields receiving the most awards


In [ ]:
top_fields = visualization_field_counts.head(15).sort_values("total_awards")

fig, ax = plt.subplots(figsize=(10.5, 8.5), layout="constrained")
bars = ax.barh(
    wrap_labels(top_fields["Field of Study"]),
    top_fields["total_awards"],
    color=COLORS["vermillion"],
    edgecolor=COLORS["black"],
    linewidth=0.8,
)
ax.bar_label(bars, padding=4, fontsize=8.5)
ax.set_title(
    "Fields receiving the most 2026 NSF GRFP awards",
    loc="left",
    weight="bold",
)
ax.set_xlabel("Number of awards")
ax.set_ylabel("")
ax.set_xlim(0, top_fields["total_awards"].max() * 1.12)
style_axis(ax)
save_figure(fig, FIGURE_DIR / "top-fields-total-awards.png")
plt.show()
plt.close(fig)


### Institutional pathways among highly awarded fields


In [ ]:
top_pathways = visualization_field_counts.head(12).sort_values("total_awards")
y = np.arange(len(top_pathways))
height = 0.34

fig, ax = plt.subplots(figsize=(10.5, 8.3), layout="constrained")
same = ax.barh(
    y - height / 2,
    top_pathways["same_institution"],
    height,
    label="Senior Undergraduates",
    color=COLORS["orange"],
    edgecolor=COLORS["black"],
    linewidth=0.8,
)
different = ax.barh(
    y + height / 2,
    top_pathways["different_institution"],
    height,
    label="1st year PhD students",
    color=COLORS["sky_blue"],
    edgecolor=COLORS["black"],
    linewidth=0.8,
)
ax.bar_label(same, padding=3, fontsize=8)
ax.bar_label(different, padding=3, fontsize=8)
ax.set_yticks(y, wrap_labels(top_pathways["Field of Study"]))
ax.set_title(
    "The proportion of senior undergraduates is not constant among top-awarded fields",
    loc="left",
    weight="bold",
)
ax.set_xlabel("Number of awards")
ax.set_ylabel("")
ax.legend(frameon=False, loc="lower right")
ax.set_xlim(
    0,
    max(
        top_pathways["same_institution"].max(),
        top_pathways["different_institution"].max(),
    )
    * 1.16,
)
style_axis(ax)
save_figure(fig, FIGURE_DIR / "top-fields-institutional-pathways.png")
plt.show()
plt.close(fig)


### Leading institutions for Chemical Engineering first-year PhD winners


In [ ]:
chemical_engineering_phd_counts = (
    df.loc[
        (df["Field of Study"] == FIELD)
        & df["Current Institution"].notna()
        & df["Current Institution"].ne(df["Baccalaureate Institution"]),
        "Current Institution",
    ]
    .value_counts()
    .head(15)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 7.5), layout="constrained")
bars = ax.barh(
    wrap_labels(chemical_engineering_phd_counts.index, width=38),
    chemical_engineering_phd_counts.values,
    color=COLORS["bluish_green"],
    edgecolor=COLORS["black"],
    linewidth=0.8,
)
ax.bar_label(bars, padding=4, fontsize=8.5)
ax.set_title(
    "Leading institutions for Chemical Engineering first-year PhD winners",
    loc="left",
    weight="bold",
)
ax.set_xlabel("Number of winners")
ax.set_ylabel("")
ax.set_xlim(0, chemical_engineering_phd_counts.max() * 1.18)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
style_axis(ax)
save_figure(
    fig,
    FIGURE_DIR / "chemical-engineering-first-year-phd-institutions.png",
)
plt.show()
plt.close(fig)
